In [0]:
# 配置：多个源库列表 和 目标库名称
source_dbs = ["catalog_southeastasia_mdm_pr.consumer_master", "catalog_southeastasia_mdm_pr.consumer_combine", 
              "catalog_southeastasia_mdm_pr.touchpoint_master", "catalog_southeastasia_mdm_pr.touchpoint_combine"]

target_db = "catalog_southeastasia_mdm_pr.pr_snapshot01_20260625"

# 1. 如果目标库不存在，先创建（只需创建一次）
spark.sql(f"CREATE DATABASE IF NOT EXISTS {target_db}")

# 2. 遍历每个源库
for source_db in source_dbs:
    # 获取该源库中的所有表名
    tables_df = spark.sql(f"SHOW TABLES IN {source_db}")
    table_list = [row.tableName for row in tables_df.collect()]
    print(f"源库 {source_db} 共发现 {len(table_list)} 张表，开始复制...")

    # 3. 遍历克隆每张表
    for table in table_list:
        print(f"正在克隆表: {source_db}.{table} -> {target_db}.{table}")
        spark.sql(f"""
            CREATE OR REPLACE TABLE {target_db}.{table} 
            DEEP CLONE {source_db}.{table}
        """)

print("所有表复制完成！")